In [79]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
from dig4bio.io import read_raman_file
from sklearn.linear_model import LinearRegression
from dig4bio.constants import FINGERPRINT_COLUMNS, LABEL_COLUMNS
from dig4bio.cv import augmented_cross_validate_sample_folds
from dig4bio.models import create_model_factory, AveragePredictor
from dig4bio.paths import EXPERIMENTS_FOLDER
from dig4bio.utils import create_new_experiment_run
from sklearn.model_selection import cross_validate, GroupKFold
import shutil

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [80]:
EXPERIMENT_FOLDER = EXPERIMENTS_FOLDER / "exp_001_baselines"
run_folder = create_new_experiment_run(EXPERIMENT_FOLDER)
shutil.copy2(EXPERIMENT_FOLDER/'notebook.ipynb',run_folder/'notebook.ipynb');

In [81]:
source_df = read_raman_file(level='processed',subfolder='source_grid_fingerprint_linear',name='source_datasets.csv')
source_df.sample(10)

,300,301,302,303,304,305,306,307,308,309,...,1797,1798,1799,1800,glucose,Na_acetate,Mg_SO4,MSM_present,fold_idx,device
858,7965.643777,7939.823276,7930.771552,7921.719828,7885.583691,7843.094421,7817.275862,7809.517241,7798.124464,7767.652361,...,1634.836257,1637.400000,1617.400000,1600.228070,0.26335,1.43570,1.441010,0.0,2,metrohm
595,3760.806500,3749.172867,3736.851333,3724.318800,3713.069033,3703.936500,3696.399800,3689.016433,3680.658200,3671.439000,...,493.252839,491.687222,489.325783,486.359987,2.13283,0.99767,0.000000,0.0,2,kaiser
1671,2350.675028,2381.940684,2354.951427,2310.942392,2255.912671,2127.233758,2133.370318,2194.450529,2305.323401,2558.366848,...,3102.048770,3101.790439,3084.652462,3072.917795,1.93507,1.46523,0.000000,0.0,4,tec
237,2948.700000,2940.385000,2932.070000,2927.380000,2922.690000,2911.000000,2899.310000,2877.765000,2856.220000,2844.455000,...,1127.035000,1128.720000,1114.195000,1099.670000,0.25636,0.67294,0.005511,0.0,4,anton532
1275,12634.991268,12609.581118,12574.267275,12540.385114,12516.592300,12511.353875,12533.195467,12585.382769,12645.081689,12685.886579,...,2274.669272,2263.364471,2277.362826,2254.937201,0.00000,0.00000,0.000000,0.0,3,mettler
113,2936.510000,2940.130000,2943.750000,2947.460000,2951.170000,2937.980000,2924.790000,2892.850000,2860.910000,2840.915000,...,1120.830000,1122.240000,1109.485000,1096.730000,0.25649,0.74089,3.345560,0.0,2,anton532
308,1011.530000,1003.650000,995.770000,993.820000,991.870000,996.730000,1001.590000,1003.250000,1004.910000,1001.515000,...,793.115000,795.190000,789.220000,783.250000,0.00000,0.76773,3.945410,0.0,0,anton785
839,3042.532189,3049.943966,3049.512931,3049.081897,3054.562232,3061.429185,3069.551724,3079.034483,3084.527897,3069.077253,...,1292.228070,1282.976471,1237.094118,1254.368421,2.93703,1.43376,0.000000,0.0,2,metrohm
1632,2364.584261,2402.649816,2376.163401,2329.870541,2269.949125,2131.632486,2131.371777,2186.918305,2293.677107,2542.609752,...,3572.662902,3569.232825,3547.093815,3529.178267,2.95207,0.00000,0.005354,0.0,3,tec
172,2925.060000,2926.070000,2927.080000,2923.655000,2920.230000,2901.995000,2883.760000,2858.385000,2833.010000,2822.725000,...,1113.470000,1115.270000,1102.650000,1090.030000,0.27710,0.69733,1.441180,0.0,3,anton532


In [82]:
transfer_plate_df = read_raman_file(level='processed',subfolder='transfer_grid_fingerprint_linear',name='transfer_plate.csv')
transfer_plate_df.sample(10)

,300,301,302,303,304,305,306,307,308,309,...,1795,1796,1797,1798,1799,1800,sample,glucose,Na_acetate,Mg_SO4
183,9483.5000,9420.472050,9465.192547,9496.57500,9514.07500,9427.552795,9336.225,9265.6,9298.329193,9371.33125,...,1658.192547,1688.006211,1694.51875,1670.14375,1695.900621,1708.1875,sample92,6.278531,0.470865,1.723417
118,9562.5000,9459.186335,9461.049689,9444.54375,9408.91875,9358.565217,9302.275,9235.4,9262.571429,9323.53125,...,1678.397516,1688.335404,1696.68750,1702.93750,1701.422360,1688.2750,sample60,3.666153,2.030915,2.488693
179,4483.1250,4466.111801,4487.229814,4484.21875,4456.09375,4414.304348,4376.000,4351.0,4401.285714,4477.32500,...,1408.677019,1400.602484,1393.04375,1386.16875,1401.496894,1402.9750,sample90,2.371494,1.454220,1.969093
150,9629.2500,9587.652174,9644.173913,9635.63750,9559.38750,9506.776398,9479.575,9510.2,9533.093168,9544.43750,...,1682.819876,1703.316770,1705.59375,1683.71875,1693.919255,1690.2250,sample76,9.704685,1.132047,2.235423
7,9574.0000,9519.633540,9605.968944,9600.69375,9500.06875,9443.670807,9384.600,9304.6,9358.826087,9461.26250,...,1672.024845,1672.645963,1673.00000,1673.00000,1685.888199,1684.2500,sample4,2.038084,0.948045,1.380240
5,9673.2500,9620.596273,9566.559006,9525.21875,9497.09375,9433.192547,9397.475,9445.6,9468.602484,9473.13750,...,1682.496894,1694.919255,1700.65625,1697.53125,1706.795031,1704.8625,sample3,3.953448,1.350473,2.132459
50,8462.4375,8444.689441,8441.583851,8427.44375,8401.81875,8373.018634,8345.350,8321.6,8343.142857,8376.96250,...,1624.795031,1644.670807,1654.11875,1649.74375,1647.453416,1632.3875,sample26,9.048045,1.532648,0.528093
156,9678.1250,9575.956522,9645.521739,9619.13750,9492.88750,9462.416149,9438.500,9401.0,9450.664596,9529.13750,...,1674.590062,1714.341615,1725.44375,1698.56875,1686.267081,1668.6875,sample79,7.429750,1.659416,1.557944
144,8966.1250,8957.242236,8969.664596,8941.08750,8869.83750,8797.322981,8743.950,8740.2,8782.925466,8837.35625,...,1651.223602,1656.813665,1661.61250,1665.36250,1660.844720,1646.7875,sample73,3.272484,2.023931,2.462035
173,9721.7500,9679.869565,9688.565217,9666.96875,9613.84375,9541.111801,9481.400,9461.4,9495.968944,9548.19375,...,1662.242236,1693.298137,1696.21875,1661.84375,1674.559006,1677.7000,sample87,11.004702,1.981495,0.409988


In [83]:
### Linear Sources: Linear regression over the source grid dataset (Cross Validated) ###

model_factory = create_model_factory(LinearRegression())

sources_lr_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False
)

print('Per Device')
display(pd.DataFrame(sources_lr_errors['devices']))
print('Overall')
display(pd.Series(sources_lr_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,-9.814037,-11.112856,-268.714326,-520.361891,-178.036716,-11937.375305,-3.594700,-2182.403212
Na_acetate,-13.009612,-0.868218,-17.183729,-67.974554,-209.363922,-6417.786683,-0.516098,-1015.802629
Mg_SO4,-5.064514,-0.036047,-9.014977,-95.357735,-259.089710,-360.502686,-0.240201,-957.250087


Overall


glucose      -2201.480448
Na_acetate   -1198.156705
Mg_SO4        -277.009218
dtype: float64

In [84]:
### Averages Sources: Analyte averages over the source grid dataset (Cross Validated) ###

model_factory = create_model_factory(AveragePredictor())

sources_avgs_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False
)

print('Per Device')
display(pd.DataFrame(sources_avgs_errors['devices']))
print('Overall')
display(pd.Series(sources_avgs_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,-0.007701,-0.008376,-0.004897,-0.004898,0.003036,-0.000517,-0.007763,-0.020530
Na_acetate,-0.027002,-0.022158,-0.053501,0.000620,-0.026815,-0.003065,-0.053188,-0.003543
Mg_SO4,-0.076486,-0.077122,-0.237535,-0.023226,-0.065905,-0.022300,-0.258090,-0.014577


Overall


glucose      -0.003056
Na_acetate   -0.003696
Mg_SO4       -0.020624
dtype: float64

In [85]:
### Linear Target: Linear regression train + test over the target device only ###

model_factory = create_model_factory(LinearRegression())

target_lr_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False,
    train_on_target= True
)

print('Per Device')
display(pd.DataFrame(target_lr_errors['devices']))
print('Overall')
display(pd.Series(target_lr_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,0.515546,0.528299,0.882741,-3.019211,0.907405,0.612470,0.517172,0.121881
Na_acetate,-0.505592,0.778544,0.821707,0.482408,0.556746,-2.572058,0.690904,-1.108420
Mg_SO4,0.549873,0.835158,0.847339,0.614719,0.879381,-2.023251,0.694120,-10.158555


Overall


glucose       0.028888
Na_acetate   -0.231723
Mg_SO4       -1.627209
dtype: float64

In [86]:
### Linear Averages: Analyte Averages train + test over the target device only ###

model_factory = create_model_factory(AveragePredictor())

target_avgs_errors = augmented_cross_validate_sample_folds(
    df = source_df,
    wavenumber_columns=FINGERPRINT_COLUMNS,
    label_columns=LABEL_COLUMNS,
    model_factory=model_factory,
    calibrate=False,
    train_on_target= True
)

print('Per Device')
display(pd.DataFrame(target_avgs_errors['devices']))
print('Overall')
display(pd.Series(target_avgs_errors['total']))

Per Device


,anton532,anton785,kaiser,metrohm,mettler,tec,timegate,tornado
glucose,-0.034112,-0.073546,-0.008200,-0.035465,-0.053714,-0.018685,-0.021394,-0.008342
Na_acetate,-0.056263,-0.023727,-0.027337,-0.020718,-0.014491,-0.030386,-0.007855,-0.022776
Mg_SO4,-0.036037,-0.067311,-0.020365,-0.039999,-0.043428,-0.017934,-0.026028,-0.034934


Overall


glucose      -0.032043
Na_acetate   -0.008138
Mg_SO4        0.000577
dtype: float64

In [87]:
model = LinearRegression()

test_df = read_raman_file(name='96_samples',level='processed',subfolder='96samples_grid_fingerprint_linear')

X = transfer_plate_df[FINGERPRINT_COLUMNS]
y = transfer_plate_df[LABEL_COLUMNS]

model.fit(X,y)
test_df[LABEL_COLUMNS] = model.predict(test_df[FINGERPRINT_COLUMNS])

display(test_df)

output_df = (
    test_df.copy()
    .groupby(by='sample',sort=False)
    .mean()
    .reset_index()
)[LABEL_COLUMNS]

output_df.index.name = 'ID'
output_df = output_df.reset_index()
output_df['ID'] +=1
output_df.to_csv(run_folder/f'{EXPERIMENT_FOLDER.name}_{str(run_folder.name[4:])}.csv',index=False)
output_df

# Public Kaggle score of this: -1.23611

C:\Users\J_Bra\AppData\Local\Temp\ipykernel_32320\2544726062.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df[LABEL_COLUMNS] = model.predict(test_df[FINGERPRINT_COLUMNS])


,300,301,302,303,304,305,306,307,308,309,...,1795,1796,1797,1798,1799,1800,sample,glucose,Na_acetate,Mg_SO4
0,9459.1250,9440.062112,9440.683230,9403.02500,9325.52500,9295.440994,9269.600,9239.6,9269.279503,9315.65000,...,1650.819876,1671.316770,1689.71875,1705.34375,1696.142857,1672.2125,sample1,1.461898,0.685178,0.353928
1,9441.6250,9411.422360,9435.645963,9429.93125,9393.05625,9312.714286,9256.075,9274.2,9299.850932,9323.63125,...,1655.919255,1678.900621,1679.10000,1649.10000,1665.136646,1674.9625,sample1,2.351013,0.583927,0.467618
2,8607.1875,8614.714286,8571.857143,8517.23125,8450.35625,8408.726708,8371.900,8339.4,8352.229814,8381.90000,...,1603.565217,1642.695652,1666.07500,1668.57500,1651.987578,1625.0000,sample2,5.581757,1.732026,1.431091
3,8560.0625,8553.944099,8543.385093,8527.58750,8506.33750,8434.900621,8377.950,8374.2,8410.590062,8458.28125,...,1609.695652,1627.086957,1630.55000,1615.55000,1632.074534,1635.8375,sample2,6.211434,1.766747,1.229222
4,9432.6875,9347.453416,9321.987578,9321.55625,9347.18125,9274.819876,9216.275,9224.4,9251.919255,9279.65625,...,1677.118012,1705.068323,1704.33750,1665.58750,1693.024845,1707.9500,sample3,4.254881,0.494294,0.999877
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,9424.8750,9449.583851,9415.422360,9380.85000,9345.85000,9276.770186,9215.950,9187.2,9226.993789,9288.96875,...,1638.571429,1652.857143,1652.13125,1631.50625,1642.950311,1645.8625,sample94,3.863245,0.210255,0.701643
188,9286.9375,9219.875776,9218.633540,9176.35000,9091.35000,9072.602484,9057.475,9030.6,9054.521739,9094.01250,...,1625.894410,1648.254658,1663.95625,1670.83125,1652.925466,1624.9625,sample95,2.050828,1.525044,0.744649
189,9425.3750,9358.006211,9348.068323,9301.35000,9216.35000,9203.683230,9197.850,9186.6,9245.931677,9329.59375,...,1638.242236,1669.298137,1678.40000,1658.40000,1670.981366,1669.5000,sample95,1.719697,1.459425,0.542773
190,9433.0625,9367.043478,9297.478261,9233.82500,9176.32500,9134.068323,9090.300,9037.8,9117.720497,9246.71875,...,1609.689441,1651.925466,1668.74375,1651.86875,1650.546584,1638.9375,sample96,4.173476,1.753767,0.483940


,ID,glucose,Na_acetate,Mg_SO4
0,1,1.906455,0.634553,0.410773
1,2,5.896595,1.749387,1.330156
2,3,4.228061,0.449639,0.878225
3,4,2.114411,1.026445,0.107280
4,5,8.702727,0.816468,0.770538
...,...,...,...,...
91,92,0.654263,1.251363,1.186624
92,93,4.352456,0.247024,0.550650
93,94,3.855683,0.297089,0.666311
94,95,1.885262,1.492234,0.643711


In [88]:
comparison = pd.DataFrame(
    (sources_avgs_errors['total'],
    sources_lr_errors['total'],
    target_avgs_errors['total'],
    target_lr_errors['total']),
).T

comparison.columns = ['Averages Sources', 'Linear Sources', 'Averages Target', 'Linear Target']
comparison

,Averages Sources,Linear Sources,Averages Target,Linear Target
glucose,-0.003056,-2201.480448,-0.032043,0.028888
Na_acetate,-0.003696,-1198.156705,-0.008138,-0.231723
Mg_SO4,-0.020624,-277.009218,0.000577,-1.627209


In [89]:
comparison.mean(axis=0)

Averages Sources      -0.009125
Linear Sources     -1225.548790
Averages Target       -0.013202
Linear Target         -0.610015
dtype: float64